# Morph: Raw Lean 4 → Liquid Haskell → Agda

This notebook is the **morph bridge** for the Foundry Intel substrate. It lifts the
provable *ground truth* (raw Lean 4) into a Liquid Haskell refinement and an Agda
sketch, with **datalog as the eggs** (the test/gate binder) and TypeScript as the
runtime glue.

## Layer stack (per the handoff)

| Layer | Tech | Role |
|-------|------|------|
| 0 | **Lean 4** (`lean-substrate/src/Substrate.lean`) | raw substrate, provable ground truth |
| 1 | **Liquid Haskell** (`packages/lh-theorems/src/Veneer/*.hs`) | refinement types over the substrate |
| 2 | **Agda** (morphed sketch, below) | dependently-typed twin of the Lean laws |
| 3 | **datalog** (`packages/datalog`) | the *eggs* — gate / test binder |
| 4 | **TypeScript** (`packages/probe-gate`) | weak-language runtime wiring |

## Hard boundaries (never violated by the morph)
- **ADR-055** RH = `OPEN_CRUX` — we prove *gate behaviour*, never RH.
- **ADR-062** = `SILENCE_PENDING` — no claim of proof authority.
- Liquid Haskell is a *refinement*, not Lean proof authority.

Run the Lean substrate proof check:
```bash
cd lean-substrate && lake env lean src/Substrate.lean   # exit 0 = all theorems proven
```

## 1. Raw Lean 4 substrate — the ground truth

The substrate proves the SYNTH gate laws and a discrete Banach contractivity seed:

- `rh_is_silence` : `assertsRh r = true → gateVerdict r = silence`  (SYNTH-008)
- `contaminated_is_silence` : `classify r = contaminated → gateVerdict r = silence`  (contractivity collapse)
- `clean_no_rh_is_evidence` / `ambiguous_no_rh_is_evidence` : clean/ambiguous + ¬RH → `evidence`
- `contractivity_fixed_point` : monotone non-increasing `f : Nat → Nat` has a fixed point (Banach seed)

The Liquid Haskell twin (`Veneer.Contractivity`) refines the *same* Banach claim with
refined types `ContractivityScore = {k | 0 < k ≤ 1}` and `τ_r = 47.06998778`.

In [ ]:
# The morph itself: read the Lean substrate and lift its theorem signatures to Agda.
# (This cell is the executable bridge — it runs on the python3 kernel.)
import re, pathlib, textwrap

LEAN_PATH = pathlib.Path("../lean-substrate/src/Substrate.lean")

LEAN_SRC = LEAN_PATH.read_text(encoding="utf-8") if LEAN_PATH.exists() else ""

# Pull every `theorem <name> ...` declaration out of the substrate.
theorems = re.findall(r"^theorem\s+(\w+)", LEAN_SRC, re.MULTILINE)
print(f"Lean substrate theorems discovered: {theorems}")

def agda_of(name: str) -> str:
    # The morph: Lean theorem name -> Agda postulate (hole left for the proof term).
    # Agda twin of the Lean law; the `?` is the dependently-typed proof obligation.
    return f"rawSubstrate-{name} : Set\nrawSubstrate-{name} = ?"

agda_module = textwrap.dedent("""\
module RawSubstrate where

-- MORPHED FROM lean-substrate/src/Substrate.lean (raw Lean 4 substrate)
-- Liquid Haskell twin: packages/lh-theorems/src/Veneer/Contractivity.hs
-- datalog (eggs): packages/datalog

data Verdict : Set where
  evidence : Verdict
  silence  : Verdict

data Class : Set where
  clean ambiguous contaminated : Class
""") + "\n".join(agda_of(t) for t in theorems) + "\n"

print("\n----- morphed Agda sketch (RawSubstrate.agda) -----")
print(agda_module)

# Persist the morph so it can be dropped into an Agda toolchain later.
out = pathlib.Path("RawSubstrate.agda")
out.write_text(agda_module, encoding="utf-8")
print(f"\n[written] {out.resolve()}")

## 2. Liquid Haskell refinement (layer 1)

The Lean `contractivity_fixed_point` seed is *refined* (not re-proven) by Liquid Haskell
in `packages/lh-theorems/src/Veneer/Contractivity.hs`:

```haskell
{-@ type ContractivityScore    = {k:Double | k > 0.0 && k <= 1.0} @-}
{-@ type StrictlyContractive  = {k:Double | k > 0.0 && k <  1.0} @-}
{-@ type TauR                 = {t:Double | t == 47.06998778}   @-}
```

Build (requires `ghc` + `liquid`):
```bash
cd packages/lh-theorems && cabal build all   # not installed in this env
```

## 3. Agda sketch (layer 2)

The cell above wrote `RawSubstrate.agda` — the dependently-typed twin of the Lean laws.
Each `rawSubstrate-<name> = ?` is the proof obligation to discharge in Agda.

```bash
agda RawSubstrate.agda   # not installed in this env
```

## 4. datalog — the eggs (layer 3)

The gate/test binder. Every verdict that the Lean substrate licenses must also satisfy
the datalog `agent_mesh.dl` contractivity rule before it is sealed to the WORM chain.
Lean proves *what may be claimed*; datalog proves *what was actually gated*.